<a href="https://colab.research.google.com/github/Bokenasubi-T/pomacea-distribution-jp/blob/main/notebooks/04_WorldClim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

04. Extract WorldClim climate data for *Pomacea canaliculata* grid cells

- Input: `data/pomacea_jp_thinned_30s.csv`, `data/pomacea_jp_thinned_5min.csv`, WorldClim v2.1 bioclimatic variables (30 s and 5 min)
- Output: `data/pomacea_jp_thinned_30s_climate.csv`, `data/pomacea_jp_thinned_5min_climate.csv`

- Variables: bio1 (annual mean temperature), bio6 (min temperature of coldest month), bio12 (annual precipitation)
- Note: WorldClim rasters are downloaded and cropped to Japan in this notebook; they are not stored in the repository.

In [7]:
from google.colab import drive
drive.mount("/content/drive")

# Project folders on Google Drive
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/pomacea"
DATA_DIR = f"{BASE_DIR}/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
#install library for reading and writing raster data
!pip install rasterio

In [9]:
# Download WorldClim v2.1 bioclim(30 s) to Colab's temporary disk, not Drive(the zip is several GB)
!wget -nc -P /content/worldclim https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_30s_bio.zip

# Unzip only the variables we use: bio1, bio6, bio12
!unzip -n -j /content/worldclim/wc2.1_30s_bio.zip wc2.1_30s_bio_1.tif wc2.1_30s_bio_6.tif wc2.1_30s_bio_12.tif -d /content/worldclim

File ‘/content/worldclim/wc2.1_30s_bio.zip’ already there; not retrieving.

Archive:  /content/worldclim/wc2.1_30s_bio.zip


In [10]:
import rasterio
from rasterio.windows import from_bounds

#Bounding box of Japan: (lon_min, lat_min, lon_max, lat_max)
BOUNDS = (122, 24, 146, 46)

for v in [1,6,12]:
  src_path = f"/content/worldclim/wc2.1_30s_bio_{v}.tif"
  out_path = f"{DATA_DIR}/wc2.1_30s_bio_{v}_japan.tif"

  with rasterio.open(src_path) as src:
    # Convert the bounding box to a pixel window and read only that part
    window = from_bounds(*BOUNDS, transform=src.transform).round_offsets().round_lengths()
    arr = src.read(1,window=window)
    profile = src.profile.copy()
    profile.update(
      height=arr.shape[0],
      width=arr.shape[1],
      transform=src.window_transform(window),
      compress="deflate"
    )

  with rasterio.open(out_path, "w", **profile) as dst:
   dst.write(arr,1)

  print(out_path, arr.shape)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_1_japan.tif (2640, 2880)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_6_japan.tif (2640, 2880)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_12_japan.tif (2640, 2880)


In [11]:
import numpy as np

def nearest_valid_cell(src, arr, lon, lat, radius=1):
    """Return (value, lon, lat) of the nearest non-NaN cell within `radius` cells."""
    row, col = src.index(lon, lat)          # convert lon/lat to row/col in the raster
    mask = np.ma.getmaskarray(arr)          # True where the cell has no data (sea)
    best, best_d = (np.nan, np.nan, np.nan), np.inf
    for dr in range(-radius, radius + 1):
        for dc in range(-radius, radius + 1):
            r, c = row + dr, col + dc
            inside = 0 <= r < arr.shape[0] and 0 <= c < arr.shape[1]
            if inside and not mask[r, c]:
                d = dr**2 + dc**2           # squared distance in cells
                if d < best_d:
                    x, y = src.xy(r, c)     # lon/lat of the cell center
                    best, best_d = (float(arr[r, c]), x, y), d
    return best



In [12]:
import pandas as pd
import rasterio

VARS = [1, 6, 12]  # bio1, bio6, bio12

pts = pd.read_csv(f"{DATA_DIR}/pomacea_jp_thinned_30s.csv")
coords = list(zip(pts["cell_lon"], pts["cell_lat"]))  # the order is (longitude, latitude)

for v in VARS:
    col = f"bio{v}"
    with rasterio.open(f"{DATA_DIR}/wc2.1_30s_bio_{v}_japan.tif") as src:
        # Extract values at cell centers; sea cells return the nodata value
        values = np.array([s[0] for s in src.sample(coords)], dtype="float64")
        values[values == src.nodata] = np.nan
        pts[col] = values

        # Mark cells to fill and prepare columns for the source cell
        pts[f"{col}_filled"] = pts[col].isna()
        pts[f"{col}_src_lon"] = np.nan
        pts[f"{col}_src_lat"] = np.nan

        # Fill NaN cells with the nearest land cell within 1 cell
        arr = src.read(1, masked=True)
        for i in pts.index[pts[f"{col}_filled"]]:
            value, x, y = nearest_valid_cell(src, arr, pts.at[i, "cell_lon"], pts.at[i, "cell_lat"])
            pts.at[i, col] = value
            pts.at[i, f"{col}_src_lon"] = x
            pts.at[i, f"{col}_src_lat"] = y

    print(f"{col}: filled {pts[f'{col}_filled'].sum()} cells, still NaN {pts[col].isna().sum()}")

bio1: filled 24 cells, still NaN 0
bio6: filled 24 cells, still NaN 0
bio12: filled 24 cells, still NaN 0


In [13]:
# All variables should share the same land/sea mask, so the filled cells and source cells should match
for v in [1, 12]:
    same_cells = (pts[f"bio{v}_filled"] == pts["bio6_filled"]).all()
    same_src = pts[f"bio{v}_src_lon"].equals(pts["bio6_src_lon"]) and pts[f"bio{v}_src_lat"].equals(pts["bio6_src_lat"])
    print(f"bio{v}: same filled cells = {same_cells}, same source cells = {same_src}")

bio1: same filled cells = True, same source cells = True
bio12: same filled cells = True, same source cells = True


In [14]:
# Show the cells with NaN values
nan_pts = pts[pts["bio6_filled"]]
print(nan_pts[["cell_lon", "cell_lat"]])

       cell_lon   cell_lat
35   136.529167  34.737500
38   139.354167  35.312500
50   130.462500  33.745833
84   130.195833  32.854167
85   130.429167  33.679167
87   129.779167  32.604167
95   130.495833  32.354167
97   131.862500  33.204167
100  131.987500  32.895833
105  130.904167  33.895833
106  130.779167  31.579167
110  130.812500  31.679167
112  130.695833  31.495833
113  130.795833  31.695833
115  130.237500  32.620833
119  131.620833  33.370833
122  131.737500  33.529167
126  129.945833  33.529167
127  131.470833  31.745833
136  130.604167  33.879167
145  134.854167  34.495833
147  131.087500  33.637500
148  130.629167  33.887500
150  129.645833  33.020833


In [15]:
import folium

# Base map with GSI pale tiles (OSM tiles are blocked in Colab)
m = folium.Map(location=[33, 131], zoom_start=6, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map",
).add_to(m)

HALF = 1 / 240  # half the width of a 30-arc-second cell in degrees

# Cells with NaN values (red point + cell boundary)
nan_layer = folium.FeatureGroup(name="Cells with NaN")
for _, r in nan_pts.iterrows():
    lat, lon = r["cell_lat"], r["cell_lon"]
    folium.Rectangle(
        bounds=[[lat - HALF, lon - HALF], [lat + HALF, lon + HALF]],
        color="red", weight=1, fill=True, fill_opacity=0.2,
    ).add_to(nan_layer)
    folium.CircleMarker(
        location=[lat, lon],
        radius=4, color="red", fill=True, fill_opacity=0.9,
        tooltip=f"lon {lon:.4f}, lat {lat:.4f}",  # show coordinates on hover
    ).add_to(nan_layer)
nan_layer.add_to(m)

# Save as HTML so the layer buttons work outside Colab too
m.save(f"{BASE_DIR}/outputs/pomacea_jp_nan_cells_map.html")
m

In [16]:
import folium

filled = pts[pts["bio6_filled"]]
HALF = 1 / 240  # half the width of a 30-arc-second cell in degrees

def cell_bounds(lon, lat):
    """Return the corners of a 30-arc-second cell for folium.Rectangle."""
    return [[lat - HALF, lon - HALF], [lat + HALF, lon + HALF]]

# Base map with GSI pale tiles
m = folium.Map(location=[filled["cell_lat"].mean(), filled["cell_lon"].mean()], zoom_start=6, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map",
).add_to(m)

for _, r in filled.iterrows():
    tip = f"bio6 = {r['bio6']:.1f} °C"

    # Original cell on the sea (red)
    folium.Rectangle(
        bounds=cell_bounds(r["cell_lon"], r["cell_lat"]),
        color="red", weight=1, fill=True, fill_opacity=0.25,
        tooltip=f"Original cell (no data)<br>{tip}",
    ).add_to(m)

    # Land cell whose value was used (blue)
    folium.Rectangle(
        bounds=cell_bounds(r["bio6_src_lon"], r["bio6_src_lat"]),
        color="blue", weight=1, fill=True, fill_opacity=0.25,
        tooltip=f"Source cell (land)<br>{tip}",
    ).add_to(m)

    # Line from the original cell to the source cell
    folium.PolyLine(
        locations=[[r["cell_lat"], r["cell_lon"]], [r["bio6_src_lat"], r["bio6_src_lon"]]],
        color="black", weight=1,
    ).add_to(m)

    # Center points drawn in screen pixels, so they stay visible at any zoom level
    folium.CircleMarker(
        location=[r["cell_lat"], r["cell_lon"]],
        radius=4, color="red", fill=True, fill_opacity=0.9,
        tooltip=f"Original cell (no data)<br>{tip}",
    ).add_to(m)
    folium.CircleMarker(
        location=[r["bio6_src_lat"], r["bio6_src_lon"]],
        radius=4, color="blue", fill=True, fill_opacity=0.9,
        tooltip=f"Source cell (land)<br>{tip}",
    ).add_to(m)

m.save(f"{BASE_DIR}/outputs/pomacea_jp_filled_cells_map.html")
m

In [17]:
out_path = f"{DATA_DIR}/pomacea_jp_thinned_30s_climate.csv"
pts.to_csv(out_path, index=False)
print(out_path, pts.shape)

/content/drive/MyDrive/Colab Notebooks/pomacea/data/pomacea_jp_thinned_30s_climate.csv (205, 29)
